In [1]:
from matlab import engine
from pyhctsa.operations.medical import hrv_classic
from pyhctsa.utils import get_dataset, z_score
import yaml
import itertools
import matlab
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score, max_error, median_absolute_error
import importlib
from pyhctsa.operations.correlation import add_noise
import os
import joblib
import numpy as np
from collections import defaultdict
from scipy.stats import pearsonr
import pandas as pd
from scipy.stats import ks_2samp

In [2]:
data = get_dataset()

Loading empirical1000 dataset...
Loaded dataset of 1000 time series.


In [3]:
from pyhctsa.calculator import FeatureCalculator

In [4]:
calc = FeatureCalculator()

Loaded 773 master operations.


In [7]:
res = calc.extract(data[1])

Evaluating 773 partialed functions. Strap in!...
Feature extraction completed in 12.371 seconds.


In [ ]:
pd.read_csv('/Users/jmoo2880/Documents/pyhctsa/tests/validation/validation_results_HCTSA.csv').drop(columns=['Unnamed: 0']).to_csv('./')

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,autocorr,tau1_zscoreTrue,tau1_zscoreTrue,CO_AutoCorr,correlation,1.000000,8.426342e-17,2.783181e-16,0,0
1,autocorr,tau2_zscoreTrue,tau2_zscoreTrue,CO_AutoCorr,correlation,1.000000,6.546616e-17,2.932692e-16,0,0
2,autocorr,tau3_zscoreTrue,tau3_zscoreTrue,CO_AutoCorr,correlation,1.000000,6.912613e-17,8.545523e-16,0,0
3,autocorr,tau4_zscoreTrue,tau4_zscoreTrue,CO_AutoCorr,correlation,1.000000,6.064069e-17,7.992631e-16,0,0
4,autocorr,tau5_zscoreTrue,tau5_zscoreTrue,CO_AutoCorr,correlation,1.000000,5.928398e-17,4.155699e-16,0,0
...,...,...,...,...,...,...,...,...,...,...
4554,add_noise,ami_methodkraskov1_extra_param4_tauac_zscoreTrue,meanch,CO_AddNoise,correlation,0.999999,1.845457e-05,3.593742e-03,0,0
4555,add_noise,ami_methodkraskov1_extra_param4_tauac_zscoreTrue,ami_at_10,CO_AddNoise,correlation,1.000000,7.228673e-15,2.892876e-12,0,0
4556,add_noise,ami_methodkraskov1_extra_param4_tauac_zscoreTrue,ami_at_15,CO_AddNoise,correlation,1.000000,7.079697e-15,5.025278e-12,0,0
4557,add_noise,ami_methodkraskov1_extra_param4_tauac_zscoreTrue,pcrossmean,CO_AddNoise,correlation,0.999885,6.122449e-05,5.250000e-04,0,0


In [6]:
actual_feats = list(res.keys())

In [114]:
eng = engine.start_matlab()

In [115]:
s = eng.genpath('/Users/jmoo2880/Documents/hctsa')
eng.addpath(s, nargout=0)
eng.javaaddpath('/Users/jmoo2880/Documents/hctsa/Toolboxes/infodynamics-dist/infodynamics.jar', nargout=0)

In [86]:
eng.MF_hmm_fit(z_score(data[0]).reshape(-1, 1), 0.7, 3)

{'Mu_1': -0.45685483927667203,
 'Mu_2': -0.40375304314238025,
 'Mu_3': 1.4770771329239367,
 'meanMu': 0.20548975016829482,
 'rangeMu': 1.9339319722006088,
 'maxMu': 1.4770771329239367,
 'minMu': -0.45685483927667203,
 'Cov': 0.40016486455361583,
 'Pmeandiag': 0.3817669621681817,
 'stdmeanP': 0.2495910796387519,
 'maxP': 0.7681724537593038,
 'meanP': 0.3333333333333333,
 'stdP': 0.2752859688295598,
 'LLtrainpersample': -1.3336264039847492,
 'nit': 27.0,
 'LLtestpersample': -1.3404775109417033,
 'LLdifference': -0.006851106956954123}

In [342]:
validate_deterministic = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation_deterministic.pkl')
validate_stochastic = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation_stochastic.pkl')

In [343]:
validate_final = pd.concat([validate_deterministic, validate_stochastic]).drop_duplicates()

In [ ]:
#1889

In [344]:
validate_final = validate_final.drop(columns=['n_stochastic_series', 'ks_pass_frac'])

In [ ]:
# validate_final[validate_final['corr'] < 0.9].to_csv('lessthan0p9.csv')

In [161]:
p1 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation.pkl')
p2 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation2.pkl')
p3 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation3.pkl')
p4 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation4.pkl')
p5 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation5.pkl')
p6 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation6.pkl')
p7 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation7.pkl')
p8 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation8.pkl')
p9 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation9.pkl')
p10 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation10.pkl')
p11 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation11.pkl')
p12 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation12.pkl')
p13 = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/validation13.pkl')

In [164]:
df_combined = pd.concat([p1, p2, p3, p4, p5, p6, p7, p8, p9, p10, p11, p12]).drop_duplicates()
#df_combined[(df_combined['corr'] < 0.9) & (df_combined['python_module'] == 'correlation')]
df_combined.shape

(4569, 12)

In [314]:
len([f for f in actual_feats if f.startswith('embed_pca')])

65

In [315]:
len([f for f in actual_feats if f.startswith('embed_pca')])

65

In [151]:
pickles = [f for f in os.listdir('./') if ('.pkl') in f]
dfs = []
for p in pickles:
    l = joblib.load(os.path.join('.', p))
    dfs.append(l)


In [152]:
df_final = pd.concat(dfs)
df_final

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,automutual_info_stats,est_methodgaussian_max_tau40.0_zscoreTrue,ami13,IN_AutoMutualInfoStats,information,1.0,3.518984e-14,8.972151e-11,NaN,0,0,0
1,automutual_info_stats,est_methodgaussian_max_tau40.0_zscoreTrue,ami19,IN_AutoMutualInfoStats,information,1.0,2.026502e-14,3.131826e-07,NaN,0,0,0
2,automutual_info_stats,est_methodgaussian_max_tau40.0_zscoreTrue,mami,IN_AutoMutualInfoStats,information,1.0,9.093015e-14,1.173891e-12,NaN,0,7,6
3,automutual_info_stats,est_methodgaussian_max_tau40.0_zscoreTrue,amiac1,IN_AutoMutualInfoStats,information,1.0,6.787132e-13,3.211759e-12,NaN,0,7,6
4,automutual_info_stats,est_methodgaussian_max_tau40.0_zscoreTrue,ami6,IN_AutoMutualInfoStats,information,1.0,1.655500e-13,5.143352e-10,NaN,0,7,2
...,...,...,...,...,...,...,...,...,...,...,...,...
523,motif_two,binarize_howmedian_zscoreTrue,hhh,SB_MotifTwo,symbolic,1.0,7.926992e-17,4.572691e-17,NaN,0,0,0
524,motif_two,binarize_howmedian_zscoreTrue,dddd,SB_MotifTwo,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
525,motif_two,binarize_howmedian_zscoreTrue,u,SB_MotifTwo,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
526,binary_stretch,stretch_whatlseq1_zscoreTrue,stretch_whatlseq1_zscoreTrue,SB_BinaryStretch,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0


In [ ]:
dfs

In [146]:
in_df1_not_df2 = df_final[~df_final['python_func'].isin(df_combined['python_func'])]
in_df1_not_df2

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml


In [51]:
eng.BF_ResetSeed('default', nargout=0)
noise = np.asarray(eng.randn(1, float(len(data[0])))).reshape(-1)
matlab_res = eng.CO_AddNoise(z_score(data[0]), 1, 'quantiles', 10, 'default', nargout=1)
python_res = add_noise(z_score(data[0]), 1, 'quantiles', 10, 0, noise)

In [52]:
python_res

{'pdec': np.float64(0.5510204081632653),
 'meanch': np.float64(7.476384702512112e-06),
 'ac1': np.float64(0.5136329535076094),
 'ac2': np.float64(0.1621494207939138),
 'firstUnder75': np.float64(0.12244897959183673),
 'firstUnder50': np.float64(3.0),
 'firstUnder25': np.float64(3.0),
 'ami_at_5': np.float64(0.004604890932556121),
 'ami_at_10': np.float64(0.0037202645432859176),
 'ami_at_15': np.float64(0.0048520942896682975),
 'ami_at_20': np.float64(0.004410009566490332),
 'pcrossmean': np.float64(0.2857142857142857),
 'fitexpa': np.float64(0.004158783607033425),
 'fitexpb': np.float64(0.04360403313370635),
 'fitexpr2': np.float64(0.13156782337008577),
 'fitexpadjr2': np.float64(0.09461326266242986),
 'fitexprmse': np.float64(0.00044081544926519926),
 'fitlina': np.float64(0.0001946843623694318),
 'fitlinb': np.float64(0.004151138318733727),
 'linfit_mse': np.float64(1.941707887036103e-07)}

In [89]:
df_combined[df_combined['python_module'] == 'medical']

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
1508,hrv_classic,zscoreTrue,pnn20,MD_hrv_classic,medical,1.000000,0.000000e+00,0.000000e+00,NaN,0,0,0
1509,hrv_classic,zscoreTrue,lfhf,MD_hrv_classic,medical,1.000000,1.754341e-07,1.281112e-11,NaN,0,0,0
1510,hrv_classic,zscoreTrue,pnn10,MD_hrv_classic,medical,1.000000,0.000000e+00,0.000000e+00,NaN,0,0,0
1511,hrv_classic,zscoreTrue,vlf,MD_hrv_classic,medical,1.000000,1.107822e-14,4.286283e-14,NaN,0,0,0
1512,hrv_classic,zscoreTrue,SD1,MD_hrv_classic,medical,1.000000,2.448330e-13,5.061622e-16,NaN,0,0,0
1513,hrv_classic,zscoreTrue,lf,MD_hrv_classic,medical,1.000000,6.903742e-15,5.714110e-13,NaN,0,0,0
1514,hrv_classic,zscoreTrue,hf,MD_hrv_classic,medical,1.000000,7.982915e-15,2.776206e-12,NaN,0,0,0
1515,hrv_classic,zscoreTrue,pnn30,MD_hrv_classic,medical,1.000000,0.000000e+00,0.000000e+00,NaN,0,0,0
1516,hrv_classic,zscoreTrue,SD2,MD_hrv_classic,medical,1.000000,7.289032e-13,6.724326e-16,NaN,0,0,0
1517,hrv_classic,zscoreTrue,pnn5,MD_hrv_classic,medical,1.000000,0.000000e+00,0.000000e+00,NaN,0,0,0


In [73]:
in_df1_not_df2

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,surprise,coarse_grain_methodquantile_memory5.0_num_grou...,mean,FC_Surprise,symbolic,0.998456,0.011543,0.064793,0.006000,1000,0,0
1,surprise,coarse_grain_methodquantile_memory5.0_num_grou...,uq,FC_Surprise,symbolic,0.983773,0.022118,0.067218,0.922000,1000,0,0
2,surprise,coarse_grain_methodquantile_memory5.0_num_grou...,min,FC_Surprise,symbolic,0.869718,0.004775,0.013498,0.985972,998,0,2
3,surprise,coarse_grain_methodquantile_memory5.0_num_grou...,median,FC_Surprise,symbolic,0.986593,0.018076,0.055342,0.939000,1000,0,0
4,surprise,coarse_grain_methodquantile_memory5.0_num_grou...,std,FC_Surprise,symbolic,0.990970,0.012364,0.056141,0.005000,1000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
35,pol_var,D6_d0.5_zscoreTrue,D6_d0.5_zscoreTrue,MD_polvar,medical,0.999992,0.000036,0.000491,NaN,0,0,0
36,pol_var,D3_d0.1_zscoreTrue,D3_d0.1_zscoreTrue,MD_polvar,medical,0.999880,0.000222,0.000792,NaN,0,0,0
37,pol_var,D4_d0.1_zscoreTrue,D4_d0.1_zscoreTrue,MD_polvar,medical,0.999941,0.000112,0.000547,NaN,0,0,0
38,pol_var,D5_d0.1_zscoreTrue,D5_d0.1_zscoreTrue,MD_polvar,medical,0.999929,0.000090,0.000562,NaN,0,0,0


In [256]:
l = joblib.load('/Users/jmoo2880/Documents/pyhctsa/validation/symbolic_validation.pkl')
#l2 = np.load('/Users/jmoo2880/Documents/pyhctsa/validation/correlation_raw_outputs.npz', allow_pickle=True)['arr_0'].tolist()

In [257]:
l

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,transition_matrix,how_to_cgquantile_num_groups2_tau1_zscoreTrue,T2,SB_TransitionMatrix,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
1,transition_matrix,how_to_cgquantile_num_groups2_tau1_zscoreTrue,maxeig,SB_TransitionMatrix,symbolic,1.0,1.282308e-17,2.551514e-17,NaN,0,0,0
2,transition_matrix,how_to_cgquantile_num_groups2_tau1_zscoreTrue,sumdiagcov,SB_TransitionMatrix,symbolic,1.0,1.317689e-18,1.660158e-17,NaN,0,0,0
3,transition_matrix,how_to_cgquantile_num_groups2_tau1_zscoreTrue,maxeigcov,SB_TransitionMatrix,symbolic,1.0,5.389822e-18,5.551096e-17,NaN,0,0,0
4,transition_matrix,how_to_cgquantile_num_groups2_tau1_zscoreTrue,T1,SB_TransitionMatrix,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
523,motif_two,binarize_howmedian_zscoreTrue,hhh,SB_MotifTwo,symbolic,1.0,7.926992e-17,4.572691e-17,NaN,0,0,0
524,motif_two,binarize_howmedian_zscoreTrue,dddd,SB_MotifTwo,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
525,motif_two,binarize_howmedian_zscoreTrue,u,SB_MotifTwo,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0
526,binary_stretch,stretch_whatlseq1_zscoreTrue,stretch_whatlseq1_zscoreTrue,SB_BinaryStretch,symbolic,1.0,0.000000e+00,0.000000e+00,NaN,0,0,0


In [253]:
l[l['corr'] < 0.9]['params'].iloc[0]

'num_surrs99_surr_methRP_the_test_statfmmi_zscoreTrue'

In [247]:
l#l[l['corr'] < 0.9]

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_prank,SD_SurrogateTest,surrogates,0.992897,0.023085,8.727912e-02,0.553,1000,0,0
1,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_f,SD_SurrogateTest,surrogates,0.946939,0.052531,2.311096e+11,0.392,1000,0,0
2,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_zscore,SD_SurrogateTest,surrogates,0.993439,33.317074,1.926721e+00,0.250,1000,0,0
3,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_p,SD_SurrogateTest,surrogates,0.992157,0.023060,2.016216e+00,0.357,1000,0,0
4,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_mediqr,SD_SurrogateTest,surrogates,0.986975,47.033138,2.750174e-01,0.283,1000,0,0


In [150]:
l2['max_tau20_what_methodols_zscoreTrue::pac_10']

{'py': array([-2.69430917e-02, -5.66765521e-03, -5.15847848e-03, -1.39283737e-02,
         1.97851960e-02,  2.38720624e-02, -1.49505295e-02,  1.61674860e-02,
        -2.30512475e-02,  8.43559017e-03, -2.81388830e-02,  5.49566166e-03,
        -1.94824140e-02,  7.39812593e-03,  7.37962048e-03, -6.84136127e-04,
         2.69888287e-02,  6.64441261e-03,  1.19450909e-02,  8.49886830e-03,
        -2.89891147e-02, -1.16183845e-02,  4.50600387e-02, -2.03856653e-02,
         9.25440813e-02, -9.79557142e-03, -1.13929482e-02, -1.30267009e-02,
         2.13650171e-02, -1.70168423e-02, -3.25701962e-02, -3.01873271e-03,
        -3.14565609e-02, -3.35699669e-04,  8.50598944e-03,  2.61889102e-02,
         7.28531930e-03,  2.15480554e-02,  1.30942477e-02,  6.33840327e-02,
         8.44544653e-03, -2.73122250e-02,  4.98039622e-01, -3.29204184e-02,
        -1.18209509e-03, -2.76355743e-02,  1.90284144e-05, -2.52516455e-02,
        -7.00251573e-02, -7.27830187e-03,  7.01859910e-03, -1.85110325e-02,
      

In [145]:
l[l['corr'] < 0.9]

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
2,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_9,CO_PartialAutoCorr,correlation,0.021606,2.856192e+21,8.568576e+21,NaN,0,0,0
3,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_8,CO_PartialAutoCorr,correlation,0.005260,5.251405e+10,1.660211e+26,NaN,0,0,0
4,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_5,CO_PartialAutoCorr,correlation,0.651409,1.448167e-02,8.351796e+11,NaN,0,0,0
6,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_7,CO_PartialAutoCorr,correlation,-0.005056,2.876961e+10,6.783608e+25,NaN,0,0,0
7,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_10,CO_PartialAutoCorr,correlation,-0.006737,1.724036e+27,7.764006e+42,NaN,0,0,0
8,partial_autocorr,max_tau20_what_methodols_zscoreTrue,pac_6,CO_PartialAutoCorr,correlation,0.086225,1.105796e+08,2.211592e+08,NaN,0,0,0


In [44]:
dataset = get_dataset(which='e1000')

Loading empirical1000 dataset...
Loaded dataset of 1000 time series.


In [194]:
from pyhctsa.operations.correlation import compare_min_ami

In [198]:
compare_min_ami(z_score(dataset[0]), 'std1', num_bins=[2, 3, 4])

{'min': np.float64(2.0),
 'max': np.float64(3.0),
 'range': np.float64(1.0),
 'median': np.float64(3.0),
 'mean': np.float64(2.6666666666666665),
 'std': np.float64(0.5773502691896258),
 'nunique': 2,
 'mode': np.float64(3.0),
 'modef': np.float64(0.6666666666666666),
 'conv4': np.float64(2.6666666666666665),
 'nlocmax': 0}

In [213]:
eng.CO_CompareMinAMI(z_score(dataset[0]).reshape(-1,1).tolist(), 'std1', matlab.double([2, 3, 4]))

Error using min
First input array is an invalid data type.

Error in CO_HistogramAMI (line 95)
        if min(y) < -1
           ^^^^^^
Error in CO_CompareMinAMI (line 78)
        amis(j) = CO_HistogramAMI(y,tauRange(j),binMethod,numBins(i));
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



MatlabExecutionError: 
  File /Users/jmoo2880/Documents/hctsa/Operations/CO_HistogramAMI.m, line 95, in CO_HistogramAMI

  File /Users/jmoo2880/Documents/hctsa/Operations/CO_CompareMinAMI.m, line 78, in CO_CompareMinAMI
First input array is an invalid data type.


In [45]:
eng = engine.start_matlab()
s = eng.genpath('/Users/jmoo2880/Documents/hctsa')
eng.addpath(s, nargout=0)
eng.javaaddpath('/Users/jmoo2880/Documents/hctsa/Toolboxes/infodynamics-dist/infodynamics.jar', nargout=0)

In [46]:
def _flatten(d, prefix=""):
    """Nested struct -> {dotted_key: scalar}."""
    flat = {}
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            flat.update(_flatten(v, key))
        else:
            flat[key] = v
    return flat


def _distribution_parity(py_dist, mat_dist):
    """Compare two per-series seed distributions for one feature."""
    py_dist = np.asarray(py_dist, float); py_dist = py_dist[np.isfinite(py_dist)]
    mat_dist = np.asarray(mat_dist, float); mat_dist = mat_dist[np.isfinite(mat_dist)]
    out = {
        "py":  np.mean(py_dist)  if py_dist.size  else np.nan,   # seed-averaged -> corr path
        "mat": np.mean(mat_dist) if mat_dist.size else np.nan,
        "std_py":  np.std(py_dist,  ddof=1) if py_dist.size  > 1 else np.nan,
        "std_mat": np.std(mat_dist, ddof=1) if mat_dist.size > 1 else np.nan,
    }
    if py_dist.size >= 2 and mat_dist.size >= 2:
        if np.ptp(py_dist) == 0 and np.ptp(mat_dist) == 0:
            out["ks_p"] = 1.0 if np.isclose(py_dist[0], mat_dist[0]) else 0.0
        else:
            out["ks_p"] = ks_2samp(py_dist, mat_dist).pvalue
    else:
        out["ks_p"] = np.nan
    return out

In [47]:
def generate_test_cases(yaml_file):
    """Expand a YAML config into a flat list of (suite_name, settings, param_set).

    Each test case may contain scalars, lists, or `!range [start, stop, step]`
    values; range/list values are expanded via Cartesian product so every
    parameter combination becomes its own case.
    """
    yaml.add_constructor("!range", lambda loader, node: range(*loader.construct_sequence(node)))

    with open(yaml_file, "r") as f:
        config = yaml.full_load(f)

    cases = []
    for suite_name, settings in config["functions"].items():
        for case in settings["test_cases"]:
            keys, values = zip(*case.items())

            # Wrap each value so itertools.product can iterate uniformly.
            expanded_values = [
                list(v) if isinstance(v, range) else [v]
                for v in values
            ]

            for combination in itertools.product(*expanded_values):
                param_set = dict(zip(keys, combination))
                cases.append((suite_name, settings, param_set))

    return cases


# ----------------------------------------------------------------------------
# Shared conversion / normalization helpers
# ----------------------------------------------------------------------------
def _to_matlab_column(arr):
    """Convert a 1-D array-like into an N×1 matlab.double.

    hctsa stores every time series as a column vector. matlab.double on a flat
    list produces a 1×N row, which silently broadcasts in any feature that
    combines `y` with an internally-built column (e.g. `q - y` in
    EX_MovingThreshold). Reshaping to (-1, 1) before tolist() yields
    [[v], [v], ...], which matlab reads as N×1.
    """
    col = np.asarray(arr, dtype=float).reshape(-1, 1)
    return matlab.double(col.tolist())


def _to_matlab_row(arr):
    """Convert a 1-D array-like into a 1×N matlab.double.

    For the rare function that explicitly wants a row vector. Opt in per-suite
    via `matlab_orientation: row` in the YAML; the default is column.
    """
    row = np.asarray(arr, dtype=float).reshape(1, -1)
    return matlab.double(row.tolist())


def _coerce_scalar(v):
    """Coerce Python scalars to MATLAB-friendly types.

    The MATLAB engine maps a Python int -> int64 and a Python float -> double.
    int64 is contagious in MATLAB arithmetic: any expression touching it returns
    int64 with integer rounding, so e.g. `1/int64(2)` -> 1 rather than 0.5. That
    silently corrupts things like `ones(numStates,1)/numStates` initializations.
    hctsa numeric args are effectively always doubles, so coerce ints to float.
    Leave bools (MATLAB logical) and everything non-numeric (strings) untouched.
    """
    if isinstance(v, bool):
        return v
    if isinstance(v, int):
        return float(v)
    return v


def _normalize(val):
    """Recursively convert MATLAB types / nested structs to plain Python types."""
    if isinstance(val, (matlab.double, matlab.single, matlab.int32, matlab.int64)):
        val = np.array(val).squeeze()
        return float(val) if val.ndim == 0 else val
    if isinstance(val, dict):
        return {k: _normalize(v) for k, v in val.items()}
    if isinstance(val, (np.ndarray, list)):
        val = np.array(val).squeeze()
        return float(val) if val.ndim == 0 else val
    return val


# ----------------------------------------------------------------------------
# Single parity test
# ----------------------------------------------------------------------------
def test_matlab_parity(matlab_engine, data, suite_name, settings, params):
    run_args = params.copy()
    should_zscore = run_args.pop("zscore", False)
    should_abs = run_args.pop("abs", False)

    test_data = z_score(data) if should_zscore else data.copy()
    if should_abs:
        test_data = np.abs(test_data)

    # Orientation of the time series handed to MATLAB. Default column (N×1),
    # which is what hctsa expects; opt into row (1×N) per-suite when needed.
    orientation = settings.get("matlab_orientation", "column")
    mat_data = (_to_matlab_row(test_data) if orientation == "row"
                else _to_matlab_column(test_data))

    arg_map = settings.get("arg_map", {})

    is_stochastic = settings.get("stochastic", False)
    seed_param = settings.get("seed_param", "random_seed")
    n_seeds = settings.get("n_seeds", 30) if is_stochastic else 1

    # A fixed seed in the YAML case shouldn't leak into the run or the case name.
    run_args.pop(seed_param, None)

    base_name_parts = []
    for k, v in sorted(params.items()):
        if k == seed_param:
            continue
        v_clean = str(v).translate(str.maketrans("", "", " []'"))
        base_name_parts.append(f"{k}{v_clean}")
    base_name = "_".join(base_name_parts)

    module = importlib.import_module(f"pyhctsa.operations.{settings['python_module']}")
    op_func = getattr(module, settings["python_func"])
    func_name = settings["matlab_func"]

    def _build_positional():
        mat_kwargs = {}
        for pk, pv in run_args.items():
            mat_name = arg_map.get(pk, pk)
            if isinstance(pv, list):
                mat_kwargs[mat_name] = matlab.double(pv)
            else:
                # Coerce int -> float so a Python int doesn't become MATLAB
                # int64 and poison downstream arithmetic.
                mat_kwargs[mat_name] = _coerce_scalar(pv)

        # MATLAB-only fixed args (e.g. extrap=[]) that the Python port doesn't
        # accept. Keyed by the MATLAB arg name; these never reach op_func, so
        # they don't pollute the Python call -- they only fill positional slots
        # so later args don't shift into the wrong place.
        for mat_name, val in settings.get("matlab_extra_args", {}).items():
            if isinstance(val, list):
                mat_kwargs[mat_name] = matlab.double(val)   # [] -> MATLAB empty
            else:
                mat_kwargs[mat_name] = _coerce_scalar(val)

        positional = [mat_data]
        for arg_name in settings.get("matlab_signature", []):
            if arg_name == "y" or arg_name == arg_map.get(seed_param, seed_param):
                continue  # seed handled via rng(), never positionally
            if arg_name in mat_kwargs:
                positional.append(mat_kwargs[arg_name])
        return positional

    def _run_once(seed):
        """One paired evaluation. Returns (py_res|None, mat_res|None, err|None)."""
        py_extra = {seed_param: seed} if seed is not None else {}
        py_res = _normalize(op_func(test_data, **run_args, **py_extra))

        if seed is not None:
            # Control randperm by seeding MATLAB's global stream. Relies on
            # BF_ResetSeed([]) being a no-op (default arg) -- verify for your hctsa.
            matlab_engine.eval(f"rng({int(seed)},'twister');", nargout=0)
        try:
            mat_res = _normalize(getattr(matlab_engine, func_name)(*_build_positional(), nargout=1))
        except matlab.engine.MatlabExecutionError as exc:
            msg = str(exc).strip()
            return py_res, None, (msg.splitlines()[-1] if msg else "MatlabExecutionError")
        return py_res, mat_res, None

    results = {}

    # ---- deterministic path (original behaviour) ----
    if not is_stochastic:
        py_res, mat_res, err = _run_once(None)
        if not isinstance(py_res, dict):
            py_res = {"output": py_res}

        def fill_nan(py_obj, key=""):
            if isinstance(py_obj, dict):
                for k, v in py_obj.items():
                    fill_nan(v, f"{key}.{k}" if key else k)
            else:
                results[f"{base_name}::{key}"] = {"py": py_obj, "mat": np.nan}

        if err is not None:
            print(f"MATLAB error for {base_name} (recording NaN): {err}")
            fill_nan(py_res)
            return results

        if not isinstance(mat_res, dict):
            mat_res = {"output": mat_res}

        def flatten_and_pair(py_obj, mat_obj, key=""):
            if isinstance(py_obj, dict) and isinstance(mat_obj, dict):
                for k in set(py_obj) | set(mat_obj):
                    nk = f"{key}.{k}" if key else k
                    if k not in py_obj:
                        results[f"{base_name}::{nk}"] = {"error": "Missing in Python"}
                    elif k not in mat_obj:
                        results[f"{base_name}::{nk}"] = {"error": "Missing in MATLAB"}
                    else:
                        flatten_and_pair(py_obj[k], mat_obj[k], nk)
            else:
                results[f"{base_name}::{key}"] = {"py": py_obj, "mat": mat_obj}

        flatten_and_pair(py_res, mat_res)
        return results

    # ---- stochastic path: many seeds, compare distributions ----
    seed_py, seed_mat, last_err = [], [], None
    for s in range(n_seeds):
        py_res, mat_res, err = _run_once(s)
        if err is not None:
            last_err = err
            continue
        if not isinstance(py_res, dict):
            py_res = {"output": py_res}
        if not isinstance(mat_res, dict):
            mat_res = {"output": mat_res}
        seed_py.append(_flatten(py_res))
        seed_mat.append(_flatten(mat_res))

    if not seed_py:  # every seed failed in MATLAB (e.g. series too short)
        print(f"MATLAB error for {base_name} (all seeds, NaN): {last_err}")
        py_once = op_func(test_data, **run_args)
        py_once = py_once if isinstance(py_once, dict) else {"output": py_once}
        for k in _flatten(_normalize(py_once)):
            results[f"{base_name}::{k}"] = {"py": np.nan, "mat": np.nan, "ks_p": np.nan}
        return results

    all_keys = set().union(*(set(d) for d in seed_py), *(set(d) for d in seed_mat))
    for k in all_keys:
        py_dist  = [d[k] for d in seed_py  if k in d]
        mat_dist = [d[k] for d in seed_mat if k in d]
        if not py_dist:
            results[f"{base_name}::{k}"] = {"error": "Missing in Python"}
        elif not mat_dist:
            results[f"{base_name}::{k}"] = {"error": "Missing in MATLAB"}
        else:
            results[f"{base_name}::{k}"] = _distribution_parity(py_dist, mat_dist)
    return results


# ----------------------------------------------------------------------------
# Correlation aggregation across the dataset
# ----------------------------------------------------------------------------
def _safe_correlation(py, mat, atol=1e-8):
    """Pearson r with fallbacks for the degenerate cases common in TS features."""
    if py.size < 2:
        return np.nan  # not enough finite points to correlate
    if np.allclose(py, mat, atol=atol):
        return 1.0  # identical constants: variance is zero but parity is perfect
    try:
        r, _ = pearsonr(py, mat)
    except ValueError:
        return np.nan  # fallback for remaining edge cases
    return r


def get_correlations(eng, data, params):
    records, raw_outputs = [], {}
    for suite_name, settings, param_dict in params:
        aggregator = defaultdict(lambda: {"py": [], "mat": [], "ks_p": []})
        for series in data:
            res = test_matlab_parity(eng, series, suite_name, settings, param_dict)
            for full_key, values in res.items():
                if "error" in values:
                    print(f"Skipping {full_key}: {values['error']}")
                    continue
                aggregator[full_key]["py"].append(values["py"])
                aggregator[full_key]["mat"].append(values["mat"])
                if "ks_p" in values and np.isfinite(values["ks_p"]):
                    aggregator[full_key]["ks_p"].append(values["ks_p"])

        for feature_name, lists in aggregator.items():
            py_vec  = np.asarray(lists["py"],  dtype=float)
            mat_vec = np.asarray(lists["mat"], dtype=float)
            raw_outputs[feature_name] = {"py": py_vec, "mat": mat_vec}

            mask = np.isfinite(py_vec) & np.isfinite(mat_vec)
            py_valid, mat_valid = py_vec[mask], mat_vec[mask]

            r = _safe_correlation(py_valid, mat_valid)
            if py_valid.size >= 1:
                mae = mean_absolute_error(py_valid, mat_valid)
                mape = mean_absolute_percentage_error(py_valid, mat_valid)
            else:
                mae = mape = np.nan

            ks = lists["ks_p"]
            ks_pass_frac = float(np.mean(np.asarray(ks) > 0.05)) if ks else np.nan

            fname = feature_name.split(":")[2]
            if fname == 'output':
                fname = feature_name.split(":")[0]
            records.append({
                "python_func": settings["python_func"],
                "params": feature_name.split(":")[0],
                "feature": fname,
                "matlab_func": settings["matlab_func"],
                "python_module": settings["python_module"],
                "corr": r, "mae": mae, "mape": mape,
                "ks_pass_frac": ks_pass_frac,
                "n_stochastic_series": len(ks) if ks else 0,
                "nan_count_py": int(np.sum(~np.isfinite(py_vec))),
                "nan_count_ml": int(np.sum(~np.isfinite(mat_vec))),
            })
            print(f"{settings['python_func']}-{feature_name}: corr={r:.4f} ks_pass={ks_pass_frac}")
    return pd.DataFrame(records), raw_outputs

In [6]:
# def generate_test_cases(yaml_file):
#     with open(yaml_file, "r") as f:
#         yaml.add_constructor('!range', lambda l, n: range(*l.construct_sequence(n)))
#         config = yaml.full_load(f)
#     cases = []
#     for suite_name, settings in config['functions'].items():
#         for case in settings['test_cases']:
#             keys, values = zip(*case.items())
#             expanded_values = []
#             for v in values:
#                 if isinstance(v, range):
#                     expanded_values.append(list(v))
#                 else:
#                     expanded_values.append([v]) # Wrap scalar/list in list for product
            
#             for combination in itertools.product(*expanded_values):
#                 param_set = dict(zip(keys, combination))
#                 cases.append((suite_name, settings, param_set))
#     return cases

In [7]:
# def _to_matlab_column(arr):
#     """Convert a 1-D array-like into an N×1 matlab.double.
 
#     hctsa stores every time series as a column vector. matlab.double on a flat
#     list produces a 1×N row, which silently broadcasts in any feature that
#     combines `y` with an internally-built column (e.g. `q - y`). Reshaping to
#     (-1, 1) before tolist() yields [[v], [v], ...], which matlab reads as N×1.
#     """
#     col = np.asarray(arr, dtype=float).reshape(-1, 1)
#     return matlab.double(col.tolist())
 
 
# def _normalize(val):
#     """Recursively convert MATLAB types / nested structs to plain Python types."""
#     if isinstance(val, (matlab.double, matlab.single, matlab.int32, matlab.int64)):
#         val = np.array(val).squeeze()
#         return float(val) if val.ndim == 0 else val
#     if isinstance(val, dict):
#         return {k: _normalize(v) for k, v in val.items()}
#     if isinstance(val, (np.ndarray, list)):
#         val = np.array(val).squeeze()
#         return float(val) if val.ndim == 0 else val
#     return val
 
 
# def test_matlab_parity(matlab_engine, data, suite_name, settings, params):
#     """Execute a single parity test between Python and MATLAB.
 
#     Returns:
#         results (dict): keys are feature names, values are dicts containing
#                         {'py': val, 'mat': val}, or {'error': reason}.
#     """
#     run_args = params.copy()
#     should_zscore = run_args.pop("zscore", False)
#     should_abs = run_args.pop("abs", False)
 
#     test_data = z_score(data) if should_zscore else data.copy()
#     if should_abs:
#         test_data = np.abs(test_data)
 
#     # MATLAB wants an N×1 column; Python keeps the native 1-D array.
#     mat_data = _to_matlab_column(test_data)
 
#     # Map Python param names to MATLAB param names.
#     arg_map = settings.get("arg_map", {})
#     mat_kwargs = {}
#     for param_key, param_val in run_args.items():
#         mat_name = arg_map.get(param_key, param_key)
#         if isinstance(param_val, list):
#             mat_kwargs[mat_name] = matlab.double(param_val)
#         else:
#             mat_kwargs[mat_name] = param_val
 
#     # Build positional args from the declared MATLAB signature (data is always 1st).
#     mat_positional_args = [mat_data]
#     for arg_name in settings.get("matlab_signature", []):
#         if arg_name == "y":
#             continue
#         if arg_name in mat_kwargs:
#             mat_positional_args.append(mat_kwargs[arg_name])
 
#     # Run both implementations.
#     module = importlib.import_module(f"pyhctsa.operations.{settings['python_module']}")
#     op_func = getattr(module, settings["python_func"])
#     py_res = _normalize(op_func(test_data, **run_args))
 
#     func_name = settings["matlab_func"]
#     mat_res = _normalize(getattr(matlab_engine, func_name)(*mat_positional_args, nargout=1))
 
#     # Build a stable base name from sorted params: "a0.1_b0.1".
#     base_name_parts = []
#     for k, v in sorted(params.items()):
#         v_clean = str(v).translate(str.maketrans("", "", " []'"))
#         base_name_parts.append(f"{k}{v_clean}")
#     base_name = "_".join(base_name_parts)
 
#     results = {}
 
#     def flatten_and_pair(py_obj, mat_obj, current_key=""):
#         """Drill into matching struct keys to produce scalar/array comparison pairs."""
#         if isinstance(py_obj, dict) and isinstance(mat_obj, dict):
#             for k in set(py_obj) | set(mat_obj):
#                 new_key = f"{current_key}.{k}" if current_key else k
#                 if k not in py_obj:
#                     results[f"{base_name}::{new_key}"] = {"error": "Missing in Python"}
#                 elif k not in mat_obj:
#                     results[f"{base_name}::{new_key}"] = {"error": "Missing in MATLAB"}
#                 else:
#                     flatten_and_pair(py_obj[k], mat_obj[k], new_key)
#         else:
#             results[f"{base_name}::{current_key}"] = {"py": py_obj, "mat": mat_obj}
 
#     # Wrap bare scalars so everything flows through the same struct-walking path.
#     if not isinstance(py_res, dict):
#         py_res = {"output": py_res}
#     if not isinstance(mat_res, dict):
#         mat_res = {"output": mat_res}
 
#     flatten_and_pair(py_res, mat_res)
#     return results

In [8]:
# def _safe_correlation(py, mat, atol=1e-8):
#     """Pearson r with fallbacks for the degenerate cases common in TS features."""
#     if py.size < 2:
#         return np.nan  # not enough finite points to correlate
#     if np.allclose(py, mat, atol=atol):
#         return 1.0  # identical constants: variance is zero but parity is perfect
#     try:
#         r, _ = pearsonr(py, mat)
#     except ValueError:
#         return np.nan  # fallback for remaining edge cases
#     return r
 
 
# def get_correlations(eng, data, params):
#     records = []
#     raw_outputs = {}
 
#     for suite_name, settings, param_dict in params:
#         aggregator = defaultdict(lambda: {"py": [], "mat": []})
 
#         for series in data:
#             res = test_matlab_parity(eng, series, suite_name, settings, param_dict)
#             for full_key, values in res.items():
#                 if "error" in values:
#                     print(f"Skipping {full_key}: {values['error']}")
#                     continue
#                 aggregator[full_key]["py"].append(values["py"])
#                 aggregator[full_key]["mat"].append(values["mat"])
 
#         for feature_name, lists in aggregator.items():
#             shapes = {np.shape(np.asarray(v, dtype=float)) for v in lists["mat"]}
#             if shapes != {()}:
#                 print(feature_name, "mat shapes:", shapes)
#             shapes_py = {np.shape(np.asarray(v, dtype=float)) for v in lists["py"]}
#             if shapes_py != {()}:
#                 print(feature_name, "py shapes:", shapes_py)
#             py_vec = np.asarray(lists["py"], dtype=float)
#             mat_vec = np.asarray(lists["mat"], dtype=float)
#             raw_outputs[feature_name] = {"py": py_vec, "mat": mat_vec}
 
#             mask = np.isfinite(py_vec) & np.isfinite(mat_vec)
#             py_valid, mat_valid = py_vec[mask], mat_vec[mask]
 
#             r = _safe_correlation(py_valid, mat_valid)
#             if py_valid.size >= 1:
#                 mae = mean_absolute_error(py_valid, mat_valid)
#                 mape = mean_absolute_percentage_error(py_valid, mat_valid)
#             else:
#                 mae = mape = np.nan
 
#             records.append({
#                 "python_func": settings["python_func"],
#                 "feature": feature_name.split(":")[2],
#                 "matlab_func": settings["matlab_func"],
#                 "python_module": settings["python_module"],
#                 "corr": r,
#                 "mae": mae,
#                 "mape": mape,
#                 "nan_count_py": int(np.sum(~np.isfinite(py_vec))),
#                 "nan_count_ml": int(np.sum(~np.isfinite(mat_vec))),
#             })
 
#             print(f"{settings['python_func']}-{feature_name}: {r:.4f}")
 
#     return pd.DataFrame(records), raw_outputs

In [80]:
params = generate_test_cases('surrogates_validation.yaml')
params

[('surrogate_test_check',
  {'python_func': 'surrogate_test',
   'python_module': 'surrogates',
   'matlab_func': 'SD_SurrogateTest',
   'arg_map': {'surr_meth': 'surrMeth',
    'num_surrs': 'numSurrs',
    'the_test_stat': 'theTestStat'},
   'matlab_extra_args': {'extrap': []},
   'matlab_signature': ['y', 'surrMeth', 'numSurrs', 'extrap', 'theTestStat'],
   'output_type': 'array',
   'stochastic': True,
   'n_seeds': 10,
   'test_cases': [{'surr_meth': 'RP',
     'num_surrs': 99,
     'the_test_stat': 'ami1',
     'zscore': True}]},
  {'surr_meth': 'RP',
   'num_surrs': 99,
   'the_test_stat': 'ami1',
   'zscore': True})]

In [83]:
df, outputs = get_correlations(eng, dataset, params)

Operation terminated by user during SD_SurrogateTest (line 137)



the MATLAB function has been cancelled
Skipping num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::output: Missing in Python


Operation terminated by user during SD_MakeSurrogates (line 118)


In SD_SurrogateTest (line 115)
z = SD_MakeSurrogates(x,surrMeth,numSurrs,extrap,randomSeed);
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



the MATLAB function has been cancelled
Skipping num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::output: Missing in Python
surrogate_test-num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::ami_zscore: corr=0.9934 ks_pass=0.25
surrogate_test-num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::ami_mediqr: corr=0.9870 ks_pass=0.282
surrogate_test-num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::ami_p: corr=0.9922 ks_pass=0.357
surrogate_test-num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::ami_prank: corr=0.9929 ks_pass=0.553
surrogate_test-num_surrs99_surr_methRP_the_test_statami1_zscoreTrue::ami_f: corr=0.9469 ks_pass=0.392


In [85]:
eng.quit()

In [38]:
from pyhctsa.operations.symbolic import surprise

In [57]:
surprise(z_score(dataset[0]), 'T1', 10, 'tau', 'embed2quadrants', 500, 1)

{'min': np.float64(0.13353139262452263),
 'max': np.float64(1.9459101490553135),
 'mean': np.float64(0.3509689360792236),
 'sum': np.float64(175.48446803961178),
 'median': np.float64(0.0),
 'lq': np.float64(-0.0),
 'uq': np.float64(0.6931471805599453),
 'std': np.float64(0.49887353583463456),
 'tstat': np.float64(29.091091713410375)}

In [84]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_zscore,SD_SurrogateTest,surrogates,0.993439,33.317074,1.926721e+00,0.250,1000,0,0
1,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_mediqr,SD_SurrogateTest,surrogates,0.986975,47.033138,2.750174e-01,0.282,1000,0,0
2,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_p,SD_SurrogateTest,surrogates,0.992157,0.023060,2.016216e+00,0.357,1000,0,0
3,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_prank,SD_SurrogateTest,surrogates,0.992897,0.023085,8.727912e-02,0.553,1000,0,0
4,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_f,SD_SurrogateTest,surrogates,0.946939,0.052531,2.311096e+11,0.392,1000,0,0


In [61]:
eng.SD_SurrogateTest(z_score(dataset[0]).reshape(-1, 1), 'RP', 99, '', 'ami1')

{'ami_p': 0.8022420196447544,
 'ami_zscore': -0.849656732014462,
 'ami_f': 0.19536859205779447,
 'ami_mediqr': 1.2313134850636027,
 'ami_prank': 0.93}

In [29]:
from pyhctsa.operations.surrogates import surrogate_test

In [64]:
surrogate_test(z_score(dataset[0]), 'RP', 99, 'ami1', 0)

{'ami_p': np.float64(0.9022255015992962),
 'ami_zscore': np.float64(-1.2943371292006005),
 'ami_f': 0.1367868776057187,
 'ami_mediqr': np.float64(1.2723425308707155),
 'ami_prank': np.float64(0.95)}

In [23]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_p,SD_SurrogateTest,surrogates,NaN,NaN,NaN,NaN,0,10,10
1,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_zscore,SD_SurrogateTest,surrogates,NaN,NaN,NaN,NaN,0,10,10
2,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_f,SD_SurrogateTest,surrogates,NaN,NaN,NaN,NaN,0,10,10
3,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_mediqr,SD_SurrogateTest,surrogates,NaN,NaN,NaN,NaN,0,10,10
4,surrogate_test,num_surrs99_surr_methRP_the_test_statami1_zsco...,ami_prank,SD_SurrogateTest,surrogates,NaN,NaN,NaN,NaN,0,10,10


In [36]:
outputs['num_states2_train_p0.8_zscoreTrue::Cov']['py'][:10]

array([0.39818197, 0.55101531, 0.50195457, 0.47725359, 0.68166559,
       0.4503819 , 0.37493166, 0.35821545, 0.46266263, 0.33975424])

In [37]:
outputs['num_states2_train_p0.8_zscoreTrue::Cov']['mat'][:10]

array([0.99683194, 0.88438266, 0.99362557, 0.48448326, 0.96931802,
       0.98339609, 0.98474309, 0.98656583, 0.46041764, 0.34058164])

In [32]:
from pyhctsa.operations.model_fit import hmm_fit

In [39]:
hmm_fit(z_score(dataset[0]), 0.8, 2)

{'Mu_1': np.float64(-0.4109799904871097),
 'Mu_2': np.float64(1.4628780381758404),
 'meanMu': np.float64(0.5259490238443654),
 'rangeMu': np.float64(1.87385802866295),
 'maxMu': np.float64(1.4628780381758404),
 'minMu': np.float64(-0.4109799904871097),
 'Cov': np.float64(0.3981819697117933),
 'Pmeandiag': np.float64(0.5043771822685591),
 'std_mean_p': np.float64(0.3947121959526117),
 'max_p': np.float64(0.7834808526436842),
 'mean_p': np.float64(0.5),
 'std_p': np.float64(0.322320789569395),
 'LLtrainpersample': -1.3348291973323223,
 'LLtestpersample': -1.3388453150742894,
 'LLdifference': -0.004016117741967085}

In [84]:
eng.MF_hmm_fit(z_score(dataset[0]).reshape(-1, 1), 0.8, 2.0, 1)

{'Mu_1': -0.41217910086747506,
 'Mu_2': 1.4601404130561515,
 'meanMu': 0.5239806560943382,
 'rangeMu': 1.8723195139236266,
 'maxMu': 1.4601404130561515,
 'minMu': -0.41217910086747506,
 'Cov': 0.39754818118490165,
 'Pmeandiag': 0.5043456789169696,
 'stdmeanP': 0.3935876953507983,
 'maxP': 0.782654207291104,
 'meanP': 0.5,
 'stdP': 0.3214021818864454,
 'LLtrainpersample': -1.334834961658112,
 'nit': 19.0,
 'LLtestpersample': -1.3388204133511517,
 'LLdifference': -0.003985451693039543}

In [23]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,hmm_fit,num_states2_train_p0.8_zscoreTrue,Pmeandiag,MF_hmm_fit,model_fit,0.612884,0.132809,0.565876,NaN,0,0,1
1,hmm_fit,num_states2_train_p0.8_zscoreTrue,rangeMu,MF_hmm_fit,model_fit,0.108686,0.852592,0.929551,NaN,0,0,1
2,hmm_fit,num_states2_train_p0.8_zscoreTrue,Cov,MF_hmm_fit,model_fit,0.489680,0.195783,0.934714,NaN,0,0,1
3,hmm_fit,num_states2_train_p0.8_zscoreTrue,LLtrainpersample,MF_hmm_fit,model_fit,0.826102,0.100594,0.140006,NaN,0,0,1
4,hmm_fit,num_states2_train_p0.8_zscoreTrue,meanMu,MF_hmm_fit,model_fit,0.326099,0.324105,5.437048,NaN,0,0,1
5,hmm_fit,num_states2_train_p0.8_zscoreTrue,minMu,MF_hmm_fit,model_fit,0.436001,0.298391,0.533737,NaN,0,0,1
6,hmm_fit,num_states2_train_p0.8_zscoreTrue,Mu_2,MF_hmm_fit,model_fit,0.173158,0.635316,1.811480,NaN,0,0,1
7,hmm_fit,num_states2_train_p0.8_zscoreTrue,maxMu,MF_hmm_fit,model_fit,0.173158,0.635316,1.811480,NaN,0,0,1
8,hmm_fit,num_states2_train_p0.8_zscoreTrue,LLtestpersample,MF_hmm_fit,model_fit,0.422470,301.109171,0.182933,NaN,0,0,1
9,hmm_fit,num_states2_train_p0.8_zscoreTrue,Mu_1,MF_hmm_fit,model_fit,0.436001,0.298391,0.533737,NaN,0,0,1


In [51]:
eng.CO_FirstMin(z_score(dataset[2]), 'ac', '', True)

3.0

In [43]:
from pyhctsa.operations.information import first_min

In [44]:
first_min(z_score(dataset[0]), 'ac')

2

In [39]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,first_min,min_whatac_zscoreTrue,min_whatac_zscoreTrue,CO_FirstMin,information,0.870538,57.741,0.551119,NaN,0,0,0


In [48]:
outputs['min_whatac_zscoreTrue::output']['py']

array([2.000e+00, 2.000e+00, 3.000e+00, 2.000e+00, 1.000e+00, 2.000e+00,
       2.000e+00, 1.000e+00, 1.000e+00, 2.000e+00, 1.000e+00, 3.000e+00,
       1.000e+00, 1.000e+00, 2.000e+00, 2.000e+00, 1.000e+00, 3.000e+00,
       2.000e+00, 1.000e+00, 2.700e+01, 1.670e+02, 6.770e+03, 1.702e+03,
       7.200e+02, 2.140e+03, 3.474e+03, 3.690e+02, 2.260e+02, 4.430e+02,
       6.210e+02, 3.150e+02, 6.070e+02, 6.580e+02, 5.440e+02, 1.600e+02,
       4.190e+02, 3.300e+02, 1.006e+03, 2.580e+03, 1.237e+03, 3.500e+01,
       1.895e+03, 3.100e+01, 1.500e+01, 1.500e+01, 1.000e+00, 1.688e+03,
       2.000e+00, 3.953e+03, 4.720e+02, 3.557e+03, 4.362e+03, 1.129e+03,
       5.510e+02, 9.700e+02, 4.519e+03, 8.380e+02, 1.000e+00, 5.000e+00,
       6.000e+00, 2.000e+00, 1.000e+00, 5.000e+00, 1.000e+00, 2.000e+00,
       1.000e+00, 2.000e+00, 1.000e+00, 3.000e+00, 2.094e+03, 2.986e+03,
       1.520e+02, 1.973e+03, 7.700e+01, 2.180e+02, 6.800e+01, 1.200e+01,
       2.500e+01, 4.700e+01, 4.000e+00, 2.300e+01, 

In [49]:
outputs['min_whatac_zscoreTrue::output']['mat']

array([1.000e+00, 1.000e+00, 1.000e+00, 2.000e+00, 1.000e+00, 1.000e+00,
       1.000e+00, 2.000e+00, 5.000e+00, 2.000e+00, 2.000e+00, 1.000e+00,
       3.000e+00, 3.000e+00, 3.000e+00, 1.000e+00, 1.000e+00, 3.000e+00,
       1.000e+00, 3.000e+00, 2.700e+01, 1.360e+02, 2.002e+03, 1.281e+03,
       3.700e+01, 1.814e+03, 2.267e+03, 3.690e+02, 2.260e+02, 2.370e+02,
       8.100e+01, 1.850e+02, 2.690e+02, 2.770e+02, 2.830e+02, 2.900e+01,
       9.600e+01, 1.740e+02, 7.060e+02, 9.900e+01, 9.030e+02, 2.500e+01,
       1.117e+03, 1.600e+01, 8.000e+00, 8.000e+00, 2.000e+00, 1.215e+03,
       1.000e+00, 1.887e+03, 1.360e+02, 2.867e+03, 2.521e+03, 8.770e+02,
       1.740e+02, 5.660e+02, 8.990e+02, 7.950e+02, 3.000e+00, 3.000e+00,
       3.000e+00, 1.000e+00, 2.000e+00, 2.000e+00, 1.000e+00, 1.000e+00,
       1.000e+00, 1.000e+00, 3.500e+01, 1.000e+00, 1.316e+03, 1.016e+03,
       9.600e+01, 1.019e+03, 5.200e+01, 2.140e+02, 4.900e+01, 1.200e+01,
       1.800e+01, 3.400e+01, 2.000e+00, 1.700e+01, 

In [9]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,ks_pass_frac,n_stochastic_series,nan_count_py,nan_count_ml
0,preproc_compare,detrend_methmedianf3_zscoreFalse,swss10_2,PP_Compare,pre_process,-0.001661,3.740545e+09,3.740545e+09,NaN,0,0,1
1,preproc_compare,detrend_methmedianf3_zscoreFalse,statav8,PP_Compare,pre_process,1.000000,4.688218e-02,1.803152e-02,NaN,0,0,0
2,preproc_compare,detrend_methmedianf3_zscoreFalse,statav10,PP_Compare,pre_process,0.999999,3.174368e-02,9.413917e-03,NaN,0,2,1
3,preproc_compare,detrend_methmedianf3_zscoreFalse,statav2,PP_Compare,pre_process,0.998950,7.718298e-01,2.014565e+12,NaN,0,2,2
4,preproc_compare,detrend_methmedianf3_zscoreFalse,statav4,PP_Compare,pre_process,0.999991,9.496158e-02,2.529813e-02,NaN,0,0,0
5,preproc_compare,detrend_methmedianf3_zscoreFalse,swss2_2,PP_Compare,pre_process,0.962030,4.439361e+00,3.762079e-02,NaN,0,2,2
6,preproc_compare,detrend_methmedianf3_zscoreFalse,swms10_1,PP_Compare,pre_process,0.185374,4.371099e+00,4.823064e-02,NaN,0,2,0
7,preproc_compare,detrend_methmedianf3_zscoreFalse,swss2_1,PP_Compare,pre_process,0.004001,3.256076e+01,4.512625e+12,NaN,0,2,2
8,preproc_compare,detrend_methmedianf3_zscoreFalse,statav6,PP_Compare,pre_process,0.998077,1.274938e+00,1.255397e-02,NaN,0,0,0
9,preproc_compare,detrend_methmedianf3_zscoreFalse,olbt_s2,PP_Compare,pre_process,0.466542,5.276231e-03,1.837395e-02,NaN,0,2,2


In [10]:
from pyhctsa.operations.symbolic import transition_matrix

In [11]:
transition_matrix(z_score(dataset[71]), 'quantile', 2, 2)

{'T1': np.float64(0.49819927971188477),
 'T2': np.float64(0.001600640256102441),
 'T3': np.float64(0.0020008003201280513),
 'T4': np.float64(0.49819927971188477),
 'ondiag': np.float64(0.9963985594237695),
 'stddiag': np.float64(0.0),
 'symdiff': np.float64(0.0008003201280512204),
 'symsumdiff': np.float64(-0.0004001600640256102),
 'stdeig': np.float64(0.00253083446191946),
 'maxeig': np.float64(0.4999888499219686),
 'mineig': np.float64(0.4964097095018009),
 'maximeig': np.float64(0.0),
 'sumdiagcov': np.float64(0.24641156983001283),
 'stdeigcov': np.float64(0.1742392919896246),
 'maxeigcov': np.float64(0.2464115698300129),
 'mineigcov': np.float64(2.1256795734068898e-17)}

In [90]:
eng.SB_TransitionMatrix(z_score(dataset[71]), 'quantile', 2, 2)

{'T1': 0.49819927971188477,
 'T2': 0.001600640256102441,
 'T3': 0.0020008003201280513,
 'T4': 0.49819927971188477,
 'ondiag': 0.9963985594237695,
 'stddiag': 0.0,
 'symdiff': 0.0008003201280512204,
 'symsumdiff': -0.0004001600640256102,
 'stdeig': 0.00253083446191946,
 'maxeig': 0.4999888499219686,
 'mineig': 0.4964097095018009,
 'maximeig': 0.0,
 'sumdiagcov': 0.24641156983001283,
 'stdeigcov': 0.17423929198962457,
 'maxeigcov': 0.24641156983001283,
 'mineigcov': -6.938893903907228e-18}

In [83]:
np.argsort(abs(outputs['how_to_cgquantile_num_groups2_tauac_zscoreTrue::T4']['py'] - outputs['how_to_cgquantile_num_groups2_tauac_zscoreTrue::T4']['mat']))[::-1]

array([ 71, 895, 980, 351, 886, 919,  29, 516, 426, 912, 911, 673, 832,
       843, 412,  49, 831, 411, 182, 178, 826, 181, 837,  21, 635, 988,
       828,  55, 671, 440, 829, 606, 438, 992, 699, 959, 471, 967, 976,
       844, 812, 821,  53, 169, 779, 966, 968, 165, 416, 811, 638, 977,
       802, 307, 915, 841, 898, 301, 603, 851, 974, 970, 437, 634, 773,
       225, 154, 174, 960, 969, 889, 772, 973, 999, 435, 963, 857, 760,
       962, 842, 921, 891, 852, 380, 922, 987, 679, 593, 620, 856, 707,
       964, 441, 955, 807, 900, 488, 867, 652, 223, 677, 956, 972, 965,
       752, 990,  74, 824, 171, 997, 227, 751, 882, 971, 574, 648, 226,
       961, 382, 305,  41, 846, 618, 619, 678, 975, 106, 978, 701, 774,
       528, 282, 896, 951, 847, 697, 698,  99, 245, 864, 600, 251, 853,
       280, 246, 194, 153, 748, 676, 238, 581, 232, 901, 858, 833, 484,
       747, 729, 892, 639, 576, 369, 591, 884, 586, 277, 799, 726, 592,
       735, 954, 140, 172, 152, 800, 776, 650, 791, 158, 885, 99

In [13]:
outputs['how_to_cgquantile_num_groups2_tauac_zscoreTrue::T4']['mat']

array([0.26245249, 0.25392539, 0.24662466, 0.24096386, 0.24224224,
       0.24924925, 0.2665066 , 0.24944989, 0.24724945, 0.25851703,
       0.20384077, 0.24284857, 0.22722272, 0.25345069, 0.24649299,
       0.250501  , 0.25525526, 0.27638191, 0.25945189, 0.25345069,
       0.26086957, 0.14285714, 0.        , 0.14285714, 0.2       ,
       0.4       , 0.25      , 0.        , 0.        , 0.33333333,
       0.        , 0.2       , 0.        , 0.        , 0.33333333,
       0.        , 0.2       , 0.33333333, 0.28571429, 0.        ,
       0.        , 0.26315789, 0.25      , 0.22789116, 0.24519231,
       0.24519231, 0.25252525, 0.25      , 0.25325325, 0.2       ,
       0.33333333, 0.33333333, 0.33333333, 0.27272727, 0.33333333,
       0.25      , 0.        , 0.25      , 0.17217217, 0.19678715,
       0.24324324, 0.2022022 , 0.16743349, 0.18127251, 0.25      ,
       0.14859438, 0.21442886, 0.20883534, 0.06640885, 0.24269903,
       0.        , 0.33333333, 0.23076923, 0.22222222, 0.23958

In [72]:
df[df['corr'] < 0.9]

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
71,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,T4,SB_TransitionMatrix,symbolic,0.814943,0.014526,1.971056e+13,0,0
72,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,T3,SB_TransitionMatrix,symbolic,0.746345,0.016230,5.670423e-02,0,0
73,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,ondiag,SB_TransitionMatrix,symbolic,0.807464,0.030918,1.426140e+13,0,0
74,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,stdeig,SB_TransitionMatrix,symbolic,0.752997,0.025304,1.801440e+12,0,0
75,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,maxeig,SB_TransitionMatrix,symbolic,0.737719,0.003344,6.341292e-03,0,0
76,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,symsumdiff,SB_TransitionMatrix,symbolic,0.719804,0.013864,4.380582e+13,0,0
77,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,maxeigcov,SB_TransitionMatrix,symbolic,0.858094,0.005545,8.670828e+11,0,0
79,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,symdiff,SB_TransitionMatrix,symbolic,0.750616,0.024247,8.761163e+13,0,0
80,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,mineig,SB_TransitionMatrix,symbolic,0.777705,0.033351,6.114265e+12,0,0
81,transition_matrix,how_to_cgquantile_num_groups2_tauac_zscoreTrue,stdeigcov,SB_TransitionMatrix,symbolic,0.858094,0.003921,6.131201e+11,0,0


In [63]:
outputs['k1_log_incTrue_q2.0_tau_step50_wtfdfa_zscoreTrue::r1_alpha']['py'][:10]

KeyError: 'k1_log_incTrue_q2.0_tau_step50_wtfdfa_zscoreTrue::r1_alpha'

In [58]:
outputs['k1_log_incTrue_q2.0_tau_step50_wtfdfa_zscoreTrue::r1_alpha']['mat'][:10]

array([0.51264847,        nan, 0.48647857,        nan,        nan,
              nan,        nan, 0.51098075, 0.53713514, 0.61742681])

In [39]:
from pyhctsa.operations.scaling import fluctuation_analysis

In [59]:
fluctuation_analysis(z_score(dataset[0]), 2.0, 'dfa', 50, 0, None, True)['r2_alpha']

np.float64(0.46990976649653465)

In [38]:
eng.SC_FluctAnal(z_score(dataset[0]), 2.0, 'dfa', 50.0, 0.0, '', True)['r2_alpha']

0.4695307464587432

In [64]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,r2_alpha,SC_FluctAnal,scaling,0.997571,0.004213,0.026334,0,35
1,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,ssr,SC_FluctAnal,scaling,0.990224,0.022524,0.194640,0,0
2,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,r2_linfitint,SC_FluctAnal,scaling,0.996922,0.024083,0.022186,0,35
3,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,resac1,SC_FluctAnal,scaling,0.989663,0.020384,0.182698,0,0
4,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,se1,SC_FluctAnal,scaling,0.987488,0.005835,0.284435,0,0
5,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,prop_r1,SC_FluctAnal,scaling,0.958940,0.012386,0.043314,0,0
6,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,r1_ssr,SC_FluctAnal,scaling,0.999587,0.000147,0.020679,0,245
7,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,r1_alpha,SC_FluctAnal,scaling,0.999889,0.003309,0.004764,0,245
8,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,meanssr,SC_FluctAnal,scaling,0.999895,0.009003,0.005674,0,0
9,fluctuation_analysis,absTrue_k2_log_incTrue_q2.0_tau_step50_wtfdfa_...,r2_se2,SC_FluctAnal,scaling,0.986089,0.001457,0.086933,0,35


In [34]:
from pyhctsa.operations.distribution import compare_ks_fit

In [43]:
compare_ks_fit(dataset[12], 'logn')

The data are not positive, but Log-Normal is a positive-only distribution.


nan

In [49]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,fast_dfa,zscoreTrue,zscoreTrue,SC_fastdfa,scaling,1.0,5.557789e-11,3.116361e-11,0,0


In [18]:
np.argsort(abs(outputs['what_distnnorm_zscoreFalse::peaksepx']['py'] - outputs['what_distnnorm_zscoreFalse::peaksepx']['mat']))[::-1]

array([243, 249, 248, 247, 251, 250, 745, 246, 901, 432, 635,  71, 643,
       950, 641, 409, 509,  54, 615, 587, 491, 495, 496, 896, 205, 171,
       284, 174, 195, 173, 355, 287, 224, 593, 225, 622, 262, 191, 230,
       206, 470, 533, 242, 294, 299, 298, 238, 320, 506, 307, 325, 261,
       272, 181, 271, 270, 263, 211, 157, 281, 212, 190, 240, 179, 162,
       594, 280, 180, 330, 260, 329, 590, 252, 347, 736, 740, 431, 351,
       427, 463, 900, 899, 424, 689, 869, 513,  51, 164, 963, 886, 887,
       511, 957, 914, 690, 457, 903, 459, 688, 971, 348, 848, 597, 968,
       601, 704, 911, 812, 956, 608, 730, 969, 967,  56, 839, 893, 959,
       891, 989, 804, 841, 964, 821, 808, 840, 845, 843, 844, 514,  22,
       822, 860,  70, 606, 862, 642,  53, 660, 829, 784, 787, 793, 918,
       607,  28, 800, 735, 851, 124, 415, 521, 788, 442, 781, 705, 448,
       777, 776, 857, 849, 916, 858, 855, 853, 161, 627, 182, 439, 445,
       399, 837, 420, 449, 703, 540, 539, 585, 658, 447, 446,   

In [19]:
outputs['what_distnnorm_zscoreFalse::output']

{'py': array([nan, nan, nan, nan, nan, nan, nan]),
 'mat': array([nan, nan, nan, nan, nan, nan, nan])}

In [14]:
outputs['what_distnnorm_zscoreFalse::peaksepx']['mat'][:10]

array([ 0.18698449,  0.04104104, -0.02526315,  0.0734693 ,  0.03723724,
       -0.07567568,  0.04249737,  0.12882247,  0.06974504, -0.94661656])

In [10]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,compare_ks_fit,what_distnnorm_zscoreFalse,adiff,DN_CompareKSFit,distribution,1.000000,0.000139,7.650381e-04,0,0
1,compare_ks_fit,what_distnnorm_zscoreFalse,peaksepx,DN_CompareKSFit,distribution,0.999999,1.491429,6.969351e+09,0,0
2,compare_ks_fit,what_distnnorm_zscoreFalse,relent,DN_CompareKSFit,distribution,1.000000,0.000083,3.626460e-03,74,89
3,compare_ks_fit,what_distnnorm_zscoreFalse,peaksepy,DN_CompareKSFit,distribution,1.000000,0.001685,4.842727e-03,0,0
4,compare_ks_fit,what_distnnorm_zscoreFalse,olapint,DN_CompareKSFit,distribution,0.999998,0.000062,2.159658e-04,0,0
5,compare_ks_fit,what_distnnorm_zscoreFalse,what_distnnorm_zscoreFalse,DN_CompareKSFit,distribution,NaN,NaN,NaN,7,7


In [21]:
outputs['the_mom11.0_zscoreFalse::output']['py'][:5]

array([ 2.38988253e-04,  1.45820203e-06, -9.05482659e-06,  1.16227920e-05,
        2.07540822e-06])

In [42]:
from pyhctsa.operations.distribution import cv

In [48]:
eng.DN_cv((dataset[0]), 2.0)

0.6044058498772076

In [47]:
cv((dataset[0]), 2)

np.float64(0.6044058498772082)

In [52]:
outputs['k2.0_zscoreFalse::output']['py'][:3]

array([0.60440585, 0.16731818, 0.09386446])

In [54]:
outputs['k2.0_zscoreFalse::output']['mat'][:3]

array([0.60440585, 0.16731818, 0.09386446])

In [53]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,cv,k1.0_zscoreFalse,k1.0_zscoreFalse,DN_cv,distribution,1.0,3.029521e-05,1.010460e-12,0,1
1,cv,k2.0_zscoreFalse,k2.0_zscoreFalse,DN_cv,distribution,1.0,1.857106e+03,2.020911e-12,0,1
2,custom_skewness,what_skewpearson_zscoreFalse,what_skewpearson_zscoreFalse,DN_CustomSkewness,distribution,1.0,2.895674e-13,1.072860e-05,0,0
3,custom_skewness,what_skewbowley_zscoreTrue,what_skewbowley_zscoreTrue,DN_CustomSkewness,distribution,1.0,2.433815e-18,1.944746e-16,1,1


In [31]:
eng.SY_KPSStest(z_score(dataset[-1]), 1.0)

{'stat': 0.1773687230303343, 'pValue': 0.024486728863624636}

In [28]:
outputs['lags1_zscoreTrue::pValue']['py']

array([0.1       , 0.1       , 0.1       , 0.1       , 0.1       ,
       0.1       , 0.1       , 0.1       , 0.1       , 0.1       ,
       0.1       , 0.1       , 0.1       , 0.1       , 0.03892881,
       0.1       , 0.1       , 0.1       , 0.1       , 0.1       ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.1       , 0.1       ,
       0.1       , 0.1       , 0.01      , 0.1       , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01      ,
       0.01      , 0.01      , 0.01      , 0.1       , 0.1       ,
       0.1       , 0.1       , 0.1       , 0.1       , 0.01      ,
       0.1       , 0.1       , 0.1       , 0.1       , 0.1       ,
       0.01      , 0.01      , 0.01      , 0.01      , 0.01   

In [29]:
outputs['lags1_zscoreTrue::pValue']['mat']

array([0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 ,
       0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.01, 0.01,
       0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
       0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.1 ,
       0.1 , 0.1 , 0.1 , 0.01, 0.1 , 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
       0.01, 0.01, 0.01, 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.01, 0.1 ,
       0.1 , 0.1 , 0.1 , 0.1 , 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
       0.01, 0.1 , 0.01, 0.1 , 0.01, 0.1 , 0.01, 0.1 , 0.1 , 0.1 , 0.1 ,
       0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 ,
       0.01, 0.01, 0.1 , 0.01, 0.1 , 0.1 , 0.01, 0.1 , 0.1 , 0.1 , 0.1 ,
       0.1 , 0.1 , 0.1 , 0.01, 0.01, 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 ,
       0.01, 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.01, 0.1 , 0.1 , 0.1 , 0.01,
       0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 ,
       0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.1 , 0.

In [180]:
np.argwhere(np.isnan(outputs['l20_segment_howfix_zscoreTrue::max']['mat']))

array([[407],
       [410],
       [411],
       [412],
       [413],
       [414],
       [417],
       [420],
       [421],
       [422],
       [426],
       [428],
       [429],
       [431],
       [434],
       [435],
       [438],
       [440],
       [441],
       [597],
       [598],
       [599],
       [600],
       [601],
       [602],
       [603],
       [604],
       [606],
       [609],
       [613],
       [615],
       [616],
       [618],
       [621],
       [647],
       [649],
       [650],
       [651],
       [652],
       [655],
       [656],
       [658],
       [674],
       [678],
       [682],
       [683],
       [686],
       [692],
       [693],
       [694],
       [695],
       [696],
       [697],
       [698],
       [699],
       [702],
       [704],
       [705],
       [706],
       [707],
       [709],
       [710],
       [711],
       [712],
       [713],
       [714],
       [715],
       [717],
       [719],
       [720],
       [723],
      

In [183]:
from pyhctsa.operations.stationarity import drifting_mean

In [9]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,drifting_mean,l20.0_segment_howfix_zscoreTrue,meanabsmaxmin,SY_DriftingMean,stationarity,1.0,1.735920e-14,1.513402e-16,0,0
1,drifting_mean,l20.0_segment_howfix_zscoreTrue,meanmaxmin,SY_DriftingMean,stationarity,1.0,8.010751e-15,4.268544e-15,0,0
2,drifting_mean,l20.0_segment_howfix_zscoreTrue,min,SY_DriftingMean,stationarity,1.0,8.743489e-15,1.754025e-16,0,0
3,drifting_mean,l20.0_segment_howfix_zscoreTrue,max,SY_DriftingMean,stationarity,1.0,1.920123e-14,1.699329e-16,0,0
4,drifting_mean,l20.0_segment_howfix_zscoreTrue,mean,SY_DriftingMean,stationarity,1.0,3.106709e-15,1.230867e+00,0,0
5,drifting_mean,l50.0_segment_howfix_zscoreTrue,meanabsmaxmin,SY_DriftingMean,stationarity,1.0,1.856846e-15,1.822449e-16,0,0
6,drifting_mean,l50.0_segment_howfix_zscoreTrue,meanmaxmin,SY_DriftingMean,stationarity,1.0,3.285533e-15,1.953057e-14,0,0
7,drifting_mean,l50.0_segment_howfix_zscoreTrue,min,SY_DriftingMean,stationarity,1.0,3.915129e-15,2.263634e-16,0,0
8,drifting_mean,l50.0_segment_howfix_zscoreTrue,max,SY_DriftingMean,stationarity,1.0,4.476155e-15,2.360243e-16,0,0
9,drifting_mean,l50.0_segment_howfix_zscoreTrue,mean,SY_DriftingMean,stationarity,1.0,1.149778e-15,2.914069e-01,0,0


In [101]:
eng.SY_SlidingWindow(z_score(dataset[410]), 'mean', 'std', 2.0, 1.0)

0.8411313477028991

In [102]:
from pyhctsa.operations.stationarity import sliding_window

In [105]:
sliding_window(z_score(dataset[410]), 'mean', 'std', 2.0, 1.0)

np.float64(0.8411313477028985)

In [99]:
np.argwhere(np.isnan(outputs['across_win_statstd_inc_move1_num_seg2_window_statmean_zscoreTrue::output']['mat']))

array([[410],
       [411],
       [412],
       [414],
       [417],
       [418],
       [422],
       [424],
       [425],
       [426],
       [428],
       [429],
       [433],
       [437],
       [438],
       [441],
       [604],
       [606],
       [619],
       [647],
       [648],
       [649],
       [650],
       [651],
       [653],
       [657],
       [658],
       [660],
       [676],
       [677],
       [678],
       [679],
       [680],
       [685],
       [690],
       [692],
       [693],
       [699],
       [700],
       [704],
       [705],
       [706],
       [707],
       [708],
       [714],
       [715],
       [716],
       [717],
       [721],
       [723],
       [726],
       [730],
       [731],
       [732],
       [733],
       [734],
       [736],
       [738],
       [739],
       [740],
       [741],
       [759],
       [762],
       [769],
       [773],
       [774],
       [775],
       [777],
       [779],
       [781],
       [782],
      

In [93]:
df

,python_func,params,feature,matlab_func,python_module,corr,mae,mape,nan_count_py,nan_count_ml
0,sliding_window,across_win_statstd_inc_move1_num_seg2_window_s...,across_win_statstd_inc_move1_num_seg2_window_s...,SY_SlidingWindow,stationarity,1.0,8.114073e-17,3.976159e-13,0,131


In [57]:
np.argwhere(np.isnan(outputs['extra_param25_what_typelen_zscoreTrue::output']['mat']))

array([[409],
       [414],
       [416],
       [418],
       [420],
       [422],
       [425],
       [426],
       [429],
       [430],
       [432],
       [435],
       [438],
       [439],
       [441],
       [442],
       [597],
       [598],
       [599],
       [600],
       [601],
       [602],
       [603],
       [605],
       [606],
       [607],
       [608],
       [612],
       [615],
       [616],
       [617],
       [619],
       [620],
       [647],
       [648],
       [652],
       [655],
       [656],
       [658],
       [659],
       [661],
       [662],
       [676],
       [677],
       [678],
       [679],
       [686],
       [687],
       [692],
       [693],
       [694],
       [700],
       [702],
       [703],
       [705],
       [707],
       [708],
       [715],
       [716],
       [717],
       [718],
       [720],
       [722],
       [723],
       [724],
       [726],
       [728],
       [729],
       [733],
       [736],
       [739],
      

In [54]:
from pyhctsa.operations.stationarity import stat_av

In [59]:
stat_av(z_score(dataset[409]), 'len', 25)

0.982770146889103

In [169]:
df.to_csv('criticality_validation.csv')

In [170]:
np.savez_compressed('criticality_validation_raw_outputs.npz', **outputs)